# Clustering-based song recommendation

**Idea.** Songs that are played by the same people belong together. We cluster songs by their listeners
with k-means, then recommend each user popular songs from the clusters they already listen to.

| Step | Choice | Course |
|---|---|---|
| Representation | each song = 0-1 vector over users (1 = played it), L2-normalised so distances follow cosine similarity | Week 6, "movies by customers" |
| Clustering | spherical k-means (own implementation, k-means++ start; centroids rescaled to length 1), on the full sparse matrix, no compression | Week 6 (k-means, cosine distance) |
| Choosing k | hit rate on separate validation users; Davies–Bouldin index reported for comparison | Week 6 |
| Recommending | score = (user's share of songs in the song's cluster) × (song popularity) | |
| Evaluation | hide 20% of each user's songs; hit rate / precision / recall @10 vs. a "most popular" baseline | |
| Interpretation | top Last.fm tags, genre and artists per cluster (`song_meta.parquet`, `song_tags.parquet`) | |

Input: `data/filtered_100_10.npz` (see README). Output: `data/clustering/`.

**Checkpoints.** The slow steps (k scan, final clustering, evaluations) save their results in `data/clustering/`
and load them on the next run, so re-running the notebook takes about a minute instead of several. Set
`RECOMPUTE = True` (or delete `data/clustering/`) after changing the data, the split or the algorithm.
`SPHERICAL = False` switches back to standard k-means; the two variants have separate checkpoint files (`_sph`).

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.preprocessing import normalize

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent   # works from repo root or notebooks/
DATA = ROOT / "data"
OUT = DATA / "clustering"
OUT.mkdir(exist_ok=True)
SEED = 2807
RECOMPUTE = False   # True: ignore saved checkpoints in data/clustering/ and recompute everything
SPHERICAL = True    # True: spherical k-means (centroids rescaled to length 1); False: standard k-means
TAG = "_sph" if SPHERICAL else ""   # keeps the checkpoints of the two variants apart


def checkpoint(path, compute):
    """Load the DataFrame saved at `path` (csv/parquet), or compute it with `compute()` and save it."""
    if path.exists() and not RECOMPUTE:
        print(f"loaded checkpoint {path.name}")
        return pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path, index_col=0)
    df = compute()
    df.to_parquet(path) if path.suffix == ".parquet" else df.to_csv(path)
    return df

## 1. Load the data and hold out 20% of each user's songs

The held-out songs are the test set for the recommendations. **The clustering only sees the training part**,
otherwise the evaluation would be leaking the answers.

In [ ]:
d = np.load(DATA / "filtered_100_10.npz")
user, song = d["user"], d["song"]
n_users, n_songs = int(user.max()) + 1, int(song.max()) + 1


def holdout_split(user, frac, seed):
    """Return a boolean mask marking `frac` of each user's rows (at least one) as test."""
    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(len(user)), user))            # rows grouped by user, shuffled within user
    counts = np.bincount(user, minlength=n_users)
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    pos = np.empty(len(user), dtype=np.int64)
    pos[order] = np.arange(len(user)) - np.repeat(starts, counts)  # position of each row within its user
    n_test = np.maximum(1, np.round(frac * counts)).astype(np.int64)
    return pos < n_test[user]


test = holdout_split(user, 0.2, SEED)
train = ~test
print(f"{n_users:,} users, {n_songs:,} songs; train rows {train.sum():,}, test rows {test.sum():,}")

## 2. Song × user matrix

In [ ]:
X = sp.csr_matrix((np.ones(train.sum(), dtype=np.float32), (song[train], user[train])),
                  shape=(n_songs, n_users))
X = normalize(X)                                   # unit-length rows: Euclidean distance ~ cosine distance
pop = np.bincount(song[train], minlength=n_songs)  # popularity = number of training listeners
print(X.shape, f"{X.nnz:,} non-zeros, density {X.nnz / (n_songs * n_users):.1e}")

## 3. k-means

Own implementation of the algorithm from Week 6, written for a sparse matrix:

- **Distances.** Rows of `X` have length 1, so $\lVert x - c \rVert^2 = 1 + \lVert c \rVert^2 - 2\,x \cdot c$. All
  distances come from one sparse × dense product `X @ C.T`; the 957k-dimensional points are never made dense.
- **Start.** k-means++: the first centroid is a random song, each next one is drawn with probability proportional
  to its squared distance from the centroids so far. This is the randomised form of the course's
  "pick the next centroid as far as possible from the chosen ones", and is less sensitive to outliers.
- **Spherical variant** (`SPHERICAL = True`). We compare songs by cosine distance (Week 6), so after each update
  the centroids are rescaled to length 1. Standard k-means keeps the plain average. On sparse data that average
  becomes a short vector near the origin when a cluster mixes unrelated songs. Every poorly fitting song is then
  close to it, which produced a catch-all cluster with 37% of the songs. With unit-length centroids the
  distance is $2 - 2\cos(x, c)$, so songs are assigned purely by cosine similarity.
- **Stop** when the squared error improves by less than 0.01%.

In [ ]:
def sq_dists(X, C):
    """Squared Euclidean distances from every (unit-length) row of X to every row of C."""
    return np.maximum(1.0 + (C ** 2).sum(axis=1) - 2.0 * (X @ C.T), 0.0)


def init_kmeanspp(X, k, rng):
    n = X.shape[0]
    C = np.empty((k, X.shape[1]), dtype=np.float32)
    C[0] = X[rng.integers(n)].toarray()
    d2 = sq_dists(X, C[:1]).ravel()
    for j in range(1, k):
        C[j] = X[rng.choice(n, p=d2 / d2.sum())].toarray()
        d2 = np.minimum(d2, sq_dists(X, C[j:j + 1]).ravel())
    return C


def centroids(X, labels, k, spherical=SPHERICAL):
    """Mean of the rows of X in each cluster; rescaled to length 1 for spherical k-means."""
    A = sp.csr_matrix((np.ones(len(labels), dtype=np.float32), (labels, np.arange(len(labels)))),
                      shape=(k, len(labels)))
    C = ((A @ X).toarray() / np.maximum(np.bincount(labels, minlength=k), 1)[:, None]).astype(np.float32)
    return normalize(C) if spherical else C


def kmeans(X, k, seed=0, max_iter=100, tol=1e-4, spherical=SPHERICAL):
    """Lloyd's k-means. Returns labels, centroids, squared error (inertia) and the number of iterations."""
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    C = init_kmeanspp(X, k, rng)
    prev = np.inf
    for it in range(1, max_iter + 1):
        D = sq_dists(X, C)
        labels = D.argmin(axis=1)
        dist = D[np.arange(n), labels]
        inertia = dist.sum()
        counts = np.bincount(labels, minlength=k)
        C = centroids(X, labels, k, spherical)       # new centroids = means (unit length if spherical)
        for j in np.flatnonzero(counts == 0):        # empty cluster: restart it at the worst-fitting song
            far = dist.argmax()
            C[j], dist[far] = X[far].toarray(), 0
        if prev - inertia < tol * prev:
            break
        prev = inertia
    D = sq_dists(X, C)
    labels = D.argmin(axis=1)
    return labels, C, D[np.arange(n), labels].sum(), it


def davies_bouldin(X, labels, C):
    """Davies–Bouldin index (Week 6): mean over clusters of max_j (S_i + S_j) / d(c_i, c_j)."""
    k = len(C)
    dist = np.sqrt(sq_dists(X, C)[np.arange(X.shape[0]), labels])
    S = np.bincount(labels, weights=dist, minlength=k) / np.maximum(np.bincount(labels, minlength=k), 1)
    G = C @ C.T
    sq = np.diag(G)
    M = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * G, 0))
    R = (S[:, None] + S[None, :]) / np.where(M > 0, M, np.inf)   # diagonal -> 0
    return R.max(axis=1).mean()

**Check against scikit-learn** on small random data: our Davies–Bouldin must equal sklearn's, and our k-means
should find the same partition (adjusted Rand index ≈ 1, as in the Week 6 exercise).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score

P, _ = make_blobs(n_samples=3000, centers=6, n_features=20, random_state=0)
P = sp.csr_matrix(normalize(P).astype(np.float32))
lab, C_, _, _ = kmeans(P, 6, seed=0, spherical=False)     # sklearn implements standard k-means
sk = KMeans(n_clusters=6, n_init=10, random_state=0).fit(P.toarray())
print(f"ARI vs sklearn: {adjusted_rand_score(lab, sk.labels_):.3f}")
print(f"Davies–Bouldin ours {davies_bouldin(P, lab, C_):.4f}  sklearn {davies_bouldin_score(P.toarray(), lab):.4f}")
lab_s, _, _, _ = kmeans(P, 6, seed=0, spherical=True)
print(f"ARI spherical vs standard on the same data: {adjusted_rand_score(lab, lab_s):.3f}")

## 4. Choosing k

The Davies–Bouldin index can't choose k here: on sparse, high-dimensional data it keeps falling as clusters get
smaller, with no clear minimum. Since the clusters exist to make recommendations, **k is chosen by the hit rate on
3,000 validation users**. These users are separate from the 10,000 users used for the final numbers in section 6,
so the final result isn't tuned on its own test users. Davies–Bouldin, the squared error and the size of the
largest cluster are kept for comparison.

The helpers below are also used by the evaluation in sections 6 and 8.

In [ ]:
N = 10                                                      # recommendations per user
train_u = sp.csr_matrix((np.ones(train.sum(), dtype=np.int8), (user[train], song[train])), shape=(n_users, n_songs))
test_u = sp.csr_matrix((np.ones(test.sum(), dtype=np.int8), (user[test], song[test])), shape=(n_users, n_songs))

eval_users = np.random.default_rng(SEED).choice(n_users, 10_000, replace=False)              # final numbers
val_users = np.random.default_rng(SEED + 1).choice(np.setdiff1d(np.arange(n_users), eval_users),
                                                   3_000, replace=False)                      # choosing k


def cluster_scores(labels_k, k, heard):
    """score(u, s) = user's share of songs in cluster(s) × popularity(s); heard songs excluded."""
    s = (np.bincount(labels_k[heard], minlength=k) / len(heard))[labels_k] * pop
    s[heard] = -1
    return s


def cluster_hit_rate(labels_k, k, users):
    hits = 0
    for u in users:
        heard = train_u[u].indices
        top = np.argpartition(-cluster_scores(labels_k, k, heard), N)[:N]
        hits += np.intersect1d(top, test_u[u].indices).size > 0
    return hits / len(users)

In [ ]:
KS = [10, 20, 30, 50, 75, 100, 150, 200, 300]


def run_scan():
    rows = []
    for k in KS:
        t0 = time.time()
        labels_k, C_k, inertia, n_it = kmeans(X, k, seed=SEED)
        rows.append({"k": k, "val_hit_rate": cluster_hit_rate(labels_k, k, val_users),
                     "davies_bouldin": davies_bouldin(X, labels_k, C_k), "inertia": inertia,
                     "largest_cluster": np.bincount(labels_k).max() / n_songs,
                     "iterations": n_it, "seconds": time.time() - t0})
        print({key: round(float(v), 4) for key, v in rows[-1].items()})
    return pd.DataFrame(rows)


scan = checkpoint(OUT / f"k_scan{TAG}.csv", run_scan).reset_index(drop=True)
scan

In [ ]:
panels = [("val_hit_rate", "Validation hit rate@10 (higher is better)"),
          ("davies_bouldin", "Davies–Bouldin index (lower is better)"),
          ("largest_cluster", "Share of songs in the largest cluster"),
          ("inertia", "Squared error (elbow)")]
fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
for ax, (col, title) in zip(axes.flat, panels):
    ax.plot(scan.k, scan[col], marker="o")
    ax.set(title=title, xlabel="k")
    ax.grid(alpha=0.3)
plt.show()

Pick k: the k with the highest validation hit rate. Override it here if a smaller k scores almost the same (fewer clusters are easier to interpret).

In [ ]:
K = int(scan.loc[scan.val_hit_rate.idxmax(), "k"])
print("K =", K)

## 5. Final clustering

Three restarts at the chosen k; keep the run with the lowest squared error (k-means only finds a local optimum).

In [ ]:
def run_final():
    runs = [kmeans(X, K, seed=SEED + r) for r in range(3)]
    print("inertia per restart:", [f"{r[2]:.1f}" for r in runs])
    print("ARI between restarts:", [f"{adjusted_rand_score(runs[0][0], r[0]):.2f}" for r in runs[1:]])
    best = min(runs, key=lambda r: r[2])[0]
    return pd.DataFrame({"song": np.arange(n_songs), "cluster": best})


labels = checkpoint(OUT / f"kmeans_k{K}{TAG}.parquet", run_final)["cluster"].to_numpy()
C = centroids(X, labels, K)
print(f"Davies–Bouldin: {davies_bouldin(X, labels, C):.3f}, "
      f"largest cluster: {np.bincount(labels).max() / n_songs:.1%} of songs")
pd.Series(np.bincount(labels)).describe()

## 6. Recommendations and evaluation

For a user, every unheard song gets the score

$$\text{score}(u, s) = \underbrace{\frac{\#\text{ of the user's songs in cluster}(s)}{\#\text{ of the user's songs}}}_{\text{how much the user likes that cluster}} \times \text{popularity}(s)$$

and the top 10 are recommended. The **baseline** recommends the 10 most popular unheard songs to everyone;
clustering is only useful if it beats that. Evaluated on the 10,000 evaluation users (not the validation users
used to pick k).

- **hit rate@10**: share of users with at least one held-out song among the 10
- **precision@10**: share of the 10 that are held-out songs
- **recall@10**: share of the user's held-out songs that are among the 10
- **coverage**: share of all songs recommended to at least one user (does it reach beyond the hits?)

In [ ]:
pop_order = np.argsort(-pop)


def recommend_popular(heard):
    out = []
    for s in pop_order:
        if s not in heard:
            out.append(s)
            if len(out) == N:
                return np.array(out)


def recommend_cluster(heard_idx):
    score = cluster_scores(labels, K, heard_idx)
    top = np.argpartition(-score, N)[:N]
    return top[np.argsort(-score[top])]


def evaluate(recommender, users):
    hits, prec, rec, seen = [], [], [], set()
    for u in users:
        heard = train_u[u].indices
        held = set(test_u[u].indices)
        recs = recommender(set(heard) if recommender is recommend_popular else heard)
        h = len(held.intersection(recs))
        hits.append(h > 0); prec.append(h / N); rec.append(h / len(held)); seen.update(recs)
    return {"hit rate@10": np.mean(hits), "precision@10": np.mean(prec), "recall@10": np.mean(rec),
            "coverage": len(seen) / n_songs}


variant = "spherical" if SPHERICAL else "standard"
results = checkpoint(OUT / f"recommendation_results_k{K}{TAG}.csv", lambda: pd.DataFrame({
    "most popular (baseline)": evaluate(recommend_popular, eval_users),
    f"k-means clusters ({variant}, k={K})": evaluate(recommend_cluster, eval_users)}).T)

# first version for comparison: standard k-means, k = 100 (if its checkpoint exists)
first = OUT / "recommendation_results_k100.csv"
if SPHERICAL and first.exists():
    results = pd.concat([results, pd.read_csv(first, index_col=0).iloc[[1]].rename(
        index=lambda _: "k-means clusters (standard, k=100, first version)")])
results.style.format("{:.3f}")

## 7. What are the clusters?

For each cluster: size, share of all plays, most common genre (tagtraum) and its share, the most common artists,
and the most **characteristic** Last.fm tags. A tag scores high if many of the cluster's songs have it *and* it
is over-represented compared with all songs: share in cluster × log₂(share in cluster / overall share), a TF.IDF-style
score (Week 1).

In [ ]:
meta = pd.read_parquet(DATA / "song_meta.parquet").assign(cluster=labels)
tags = pd.read_parquet(DATA / "song_tags.parquet")
tags = tags[tags.weight >= 20]                                             # ignore very weak tags
tags = tags.assign(cluster=labels[tags.song.to_numpy()])

size = np.bincount(labels, minlength=K)
share_in = tags.groupby(["cluster", "tag"]).song.nunique().div(pd.Series(size), level="cluster")
share_all = tags.groupby("tag").song.nunique() / n_songs
score = share_in * np.log2(share_in / share_all.reindex(share_in.index.get_level_values("tag")).to_numpy())
top_tags = score.groupby("cluster", group_keys=False).nlargest(5).reset_index().groupby("cluster").tag.agg(", ".join)

genre = meta.dropna(subset=["genre"]).groupby("cluster").genre.agg(
    lambda g: f"{g.mode().iloc[0]} ({(g == g.mode().iloc[0]).mean():.0%})")
artists = meta.dropna(subset=["artist"]).groupby("cluster").artist.agg(
    lambda a: ", ".join(a.value_counts().index[:3]))

clusters = pd.DataFrame({"songs": size,
                         "share of plays": meta.groupby("cluster").total_plays.sum() / meta.total_plays.sum(),
                         "main genre": genre, "top artists": artists, "characteristic tags": top_tags})
clusters.to_csv(OUT / f"clusters_k{K}{TAG}.csv")
pd.set_option("display.max_colwidth", 80)
clusters.sort_values("songs", ascending=False).style.format({"share of plays": "{:.1%}"})

## 8. Listening clusters vs. lyrics topics (LDA)

The LDA part of the project gives each song with lyrics a topic mix (θ, sums to 1). Three questions:

1. **Do listening clusters line up with lyrical themes?** Agreement (NMI, ARI) between each song's cluster and its
   dominant topic, next to genre for context.
2. **Which clusters lean towards which topics?** Average topic mix per cluster compared with the average song.
3. **Which recommends better?** The same "affinity × popularity" recommender, once with cluster affinity and once
   with lyrical similarity, on the songs that have lyrics.

`LDA_DIR` points to one LDA run (`scripts/run_lda.py`). Its `song` column is the index into *all* songs
(`old_song` in `filtered_100_10.npz`), so it is converted to this notebook's index first.

In [ ]:
from sklearn.metrics import normalized_mutual_info_score

LDA_DIR = DATA / "sample_lda_output" / "k15_seed2807"
topics = pd.read_parquet(LDA_DIR / "song_topics.parquet")
top_words = pd.read_csv(LDA_DIR / "top_words.csv")
T = topics.filter(like="topic_").shape[1]

old_song = d["old_song"]                                     # filtered index -> index into all songs (sorted)
pos = np.searchsorted(old_song, topics.song.to_numpy())
assert (old_song[np.minimum(pos, n_songs - 1)] == topics.song.to_numpy()).all(), "LDA song not in filtered set"

theta = np.zeros((n_songs, T))
theta[pos] = topics.filter(like="topic_").to_numpy()
has_lyrics = np.zeros(n_songs, dtype=bool)
has_lyrics[pos] = True
dominant = theta.argmax(axis=1)
topic_names = [f"{t}: " + " ".join(w.split()[:3]) for t, w in zip(top_words.topic, top_words.top_words)]

print(f"songs with lyrics: {has_lyrics.sum():,} / {n_songs:,} ({has_lyrics.mean():.0%}), "
      f"{meta.total_plays[has_lyrics].sum() / meta.total_plays.sum():.0%} of plays")
top_words.assign(name=topic_names)[["name", "share", "top_words"]]

### 8.1 Agreement

NMI (0 = unrelated, 1 = identical grouping) and ARI (0 = chance level) on songs that have both lyrics and a genre.
Clusters have k groups, topics and genres about 15, so ARI is pulled down by the different sizes; NMI is the fairer
number for comparing rows.

In [ ]:
genre_codes = pd.factorize(meta.genre)[0]
m = has_lyrics & meta.genre.notna().to_numpy()
pairs = {f"listening clusters (k={K}) vs lyrics topic": (labels, dominant),
         "genre vs lyrics topic": (genre_codes, dominant),
         f"listening clusters (k={K}) vs genre": (labels, genre_codes)}
agreement = pd.DataFrame({name: {"NMI": normalized_mutual_info_score(a[m], b[m]), "ARI": adjusted_rand_score(a[m], b[m])}
                          for name, (a, b) in pairs.items()}).T
print(f"{m.sum():,} songs with lyrics and genre")
agreement.style.format("{:.3f}")

### 8.2 Which listening clusters lean towards which topics?

Each cell: the cluster's average weight on a topic compared with the average song with lyrics, on a log₂ scale
(+1 = twice as much, −1 = half). Only songs with lyrics count.

With many small clusters, the largest clusters are mostly variations of rock and pop, so rows are chosen **per
topic**: for each topic, the 2 clusters that lean most towards it (among clusters with at least 80 songs with
lyrics, so their averages aren't noise). The rows are ordered by topic, so the strong cells run along the
diagonal. A weak diagonal cell means that no listening community stands out for that topic.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

MIN_LYRICS_SONGS = 80   # clusters with fewer songs with lyrics are left out (noisy averages)
PER_TOPIC = 2           # clusters shown per topic

lyr = np.flatnonzero(has_lyrics)
profile = pd.DataFrame(theta[lyr]).groupby(labels[lyr]).mean()
lift = np.log2(profile / theta[lyr].mean(axis=0))
n_lyr = np.bincount(labels[lyr], minlength=K)
eligible = lift.index[n_lyr[lift.index] >= MIN_LYRICS_SONGS]

show = []
for t in range(T):
    show += [c for c in lift.loc[eligible, t].nlargest(PER_TOPIC).index if c not in show]
row_names = [f"{c}: {', '.join(clusters.loc[c, 'characteristic tags'].split(', ')[:2])}  ({n_lyr[c]} songs)"
             for c in show]
print(f"{len(eligible)} of {K} clusters have ≥ {MIN_LYRICS_SONGS} songs with lyrics; showing {len(show)}")

diverging = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])
fig, ax = plt.subplots(figsize=(12, 0.32 * len(show) + 2.5), constrained_layout=True)
im = ax.imshow(lift.loc[show], cmap=diverging, norm=TwoSlopeNorm(0, -1.5, 1.5), aspect="auto")
ax.set_xticks(range(T), topic_names, rotation=45, ha="right")
ax.set_yticks(range(len(show)), row_names)
ax.set_title("The listening clusters that lean most towards each lyrics topic (log₂ vs. average song)", loc="left")
fig.colorbar(im, ax=ax, shrink=0.6, label="log₂(cluster mean / overall mean)")
plt.show()

### 8.3 Recommending with clusters vs. with lyrics

Same users and hold-out as section 6, but restricted to songs with lyrics (LDA can only recommend those): every
method may only recommend lyrics songs, and only held-out lyrics songs count as hits. Users need at least one
heard and one held-out lyrics song.

| Method | score(u, s) |
|---|---|
| most popular | popularity(s) |
| k-means clusters | share of user's songs in cluster(s) × popularity(s) |
| lyrics × popularity | cos(user's average topic mix, θ_s) × popularity(s) |
| lyrics only | cos(user's average topic mix, θ_s) |

In [ ]:
theta_n = theta / np.maximum(np.linalg.norm(theta, axis=1, keepdims=True), 1e-12)


def scores(method, heard):
    if method == "most popular":
        s = pop.astype(float)
    elif method == "k-means clusters":
        s = (np.bincount(labels[heard], minlength=K) / len(heard))[labels] * pop
    else:
        prof = theta[heard[has_lyrics[heard]]].mean(axis=0)
        sim = theta_n @ (prof / np.linalg.norm(prof))
        s = sim * pop if method == "lyrics × popularity" else sim
    s = s.astype(float)
    s[~has_lyrics] = -np.inf
    s[heard] = -np.inf
    return s


methods = ["most popular", "k-means clusters", "lyrics × popularity", "lyrics only"]


def run_lyrics_eval():
    res = {mth: {"hits": [], "prec": [], "rec": [], "seen": set()} for mth in methods}
    n_used = 0
    for u in eval_users:
        heard = train_u[u].indices
        held = test_u[u].indices
        held = set(held[has_lyrics[held]])
        if not held or not has_lyrics[heard].any():
            continue
        n_used += 1
        for mth in methods:
            s = scores(mth, heard)
            top = np.argpartition(-s, N)[:N]
            h = len(held.intersection(top))
            r = res[mth]
            r["hits"].append(h > 0); r["prec"].append(h / N); r["rec"].append(h / len(held)); r["seen"].update(top)

    return pd.DataFrame({mth: {"hit rate@10": np.mean(r["hits"]), "precision@10": np.mean(r["prec"]),
                                         "recall@10": np.mean(r["rec"]), "coverage": len(r["seen"]) / has_lyrics.sum()}
                                   for mth, r in res.items()}).T.assign(users=n_used)


lyrics_results = checkpoint(OUT / f"lyrics_vs_clusters_k{K}{TAG}_{LDA_DIR.name}.csv", run_lyrics_eval)
print(f"{int(lyrics_results.users.iloc[0]):,} of {len(eval_users):,} users evaluated")
lyrics_results.drop(columns="users").style.format("{:.3f}")